In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [13]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI     # 서버에 올라온 Gemma 4를 사용하기 위한 dummy
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# middleware
from collections.abc import Callable
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools.tool_node import ToolCallRequest
from langgraph.types import Command

In [ ]:
# load
loader = PyMuPDFLoader('./data/wt_data.pdf')
docs = loader.load()    # Document 형태로 만들기 위해서 사용

In [4]:
# split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

split_docs = splitter.split_documents(docs)

In [ ]:
# embed
embeddings = OpenAIEmbeddings()

In [6]:
# store
vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name='student_support',
    persist_directory='./student_chromadb'
)

In [7]:
# retrieve
retriever = vectorstore.as_retriever(
    search_kwargs={'k': 5}
)

In [8]:
# test retriever
retriever.invoke('상담시간 알려주세요.')

[Document(id='26c259f7-7e8d-408e-9eea-c4083c59a80d', metadata={'creationDate': "D:20260414083437+00'00'", 'creator': '(unspecified)', 'page': 1, 'keywords': '', 'source': './data/wt_data.pdf', 'format': 'PDF 1.4', 'trapped': '', 'total_pages': 6, 'modDate': "D:20260414083437+00'00'", 'moddate': '2026-04-14T08:34:37+00:00', 'creationdate': '2026-04-14T08:34:37+00:00', 'file_path': './data/wt_data.pdf', 'producer': 'ReportLab PDF Library - (opensource)', 'author': '(anonymous)', 'title': '(anonymous)', 'subject': '(unspecified)'}, page_content='방학 중 평일\n10:00 - 17:00\n매주 수요일은 16:00 종료\n토요일\n09:30 - 12:30\n학기 중에만 운영\n일요일 및 공휴일\n운영하지 않음\n온라인 FAQ만 이용 가능\n단, 시험기간 마지막 2주에는 학업상담 수요가 증가하므로 야간상담을 화요일과 목요일에 19:30까지 한시적으로\n운영한다. 야간상담은 사전 예약자만 가능하며 당일 접수는 불가하다.\n1-2. 위치와 연락처\n학생회관 2층 204호에 위치하며, 대표전화는 02-000-1234, 대표 이메일은 support@example.ac.kr 이다.\n심리상담실은 같은 층 209호에 별도로 있다.'),
 Document(id='f086c699-7655-4048-b7a6-bfeda69331c1', metadata={'creator': '(unspecified)', 'creationdate': '2026-04-14T08:3

In [40]:
@tool
def retrieve_context(query: str) -> tuple[str, list]:
    '''학교에 관련된 정보를 검색합니다.'''
    retrieved_docs = retriever.invoke(query)
    
    serialized = "\n\n".join(
        f"[문서 {i+1}]\n"
        f"source: {doc.metadata.get('source')}\n"
        f"page: {doc.metadata.get('page')}\n"
        f"content: {doc.page_content}"
        for i, doc in enumerate(retrieved_docs)
    )
    
    return serialized, retrieved_docs


# log 찍는 middleware
@wrap_tool_call
def log_tool_calls(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    tool_name = request.tool_call['name']
    tool_args = request.tool_call['args']
    
    print(f"[TOOL START]{tool_name}/args={tool_args}")
    
    try:
        result = handler(request)

        if isinstance(result, ToolMessage):
            print(f"[TOOL END]{tool_name}/result={result.content}")
        else:
            print(f"[TOOL END]{tool_name}/result={result}")

        return result

    except Exception as e:
        print(f"[TOOL ERROR]{tool_name}/error={e}")
        raise

In [44]:
agent = create_agent(
    model='openai:gpt-5.4-mini',
    tools=[retrieve_context],
    middleware=[log_tool_calls],
    system_prompt=(
        '당신은 학생지원센터 안내를 담당하고 있는 에이전트입니다.'
        '반드시 `retrieve_context` 도구를 먼저 사용하여 관련 내용을 문서에서 찾은 뒤 답변하세요.'
        '문서에 없는 내용은 추측해서 답변하지 말고 "음... 난 잘 알지 못해요"라고 답하세요.'
        '가능하면 답변 끝에 참고한 페이지 번호등의 출처를 명시하세요.'
    )
)

In [45]:
response = agent.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': "상담 예약은 언제까지 취소할 수 있나요?"
            }
        ]
    }
)

print(f"{response['messages'][-1].content}")

[TOOL START]retrieve_context/args={'query': '상담 예약 취소 가능 시기 상담 예약은 언제까지 취소할 수 있나요 학생지원센터'}
[TOOL END]retrieve_context/result=('[문서 1]\nsource: ./data/wt_data.pdf\npage: 2\ncontent: Page 3\nRAG Practice PDF Sample\n2. 상담 예약 및 취소 규정\n상담은 온라인 예약 시스템을 통해 신청하는 것을 원칙으로 하며, 당일 방문 접수는 남는 시간대가 있을 경우에만\n가능하다. 예약은 상담 시작 2시간 전까지 할 수 있다.\n2-1. 일반 상담 예약\n•\n학업상담과 진로상담은 1회 30분 기준이다.\n•\n심리상담 초기면담은 50분 기준이며, 후속 회기는 담당자 판단에 따라 조정된다.\n•\n하루에 동일 학생이 예약할 수 있는 상담은 최대 2건이다.\n•\n예약 확정 문자는 상담 시작 1시간 전에 자동 발송된다.\n2-2. 취소 및 노쇼 정책\n항목\n기준\n조치\n일반 취소\n상담 시작 4시간 전까지\n불이익 없음\n지연 취소\n상담 시작 4시간 전 이후 ~ 시작 전\n월 2회까지 경고 없음\n노쇼\n사전 연락 없이 미참석\n해당 학기 온라인 예약 2주 제한\n반복 노쇼\n같은 학기 2회 이상\n상담사 사전 승인 후에만 예약 가능\n예외 조항\n\n[문서 2]\nsource: ./data/wt_data.pdf\npage: 1\ncontent: 방학 중 평일\n10:00 - 17:00\n매주 수요일은 16:00 종료\n토요일\n09:30 - 12:30\n학기 중에만 운영\n일요일 및 공휴일\n운영하지 않음\n온라인 FAQ만 이용 가능\n단, 시험기간 마지막 2주에는 학업상담 수요가 증가하므로 야간상담을 화요일과 목요일에 19:30까지 한시적으로\n운영한다. 야간상담은 사전 예약자만 가능하며 당일 접수는 불가하다.\n1-2. 위치와 연락처\n학생회관 2층 204호에 위치하며, 대표전화는 02-000-123

In [73]:
model = ChatOpenAI(
    model = 'google/gemma-4-26B-A4B-it',
    base_url="http://100.95.34.69:8001/v1",
    api_key='dummy',
)

agent = create_agent(
    model=model, 
    tools=[retrieve_context],
    middleware=[log_tool_calls],
    system_prompt=(
        '당신은 학생지원센터 안내를 담당하고 있는 에이전트입니다.'
        '반드시 `retrieve_context` 도구를 먼저 사용하여 관련 내용을 문서에서 찾은 뒤 답변하세요.'
        '문서에 없는 내용은 추측해서 답변하지 말고 "음... 난 잘 알지 못해요"라고 답하세요.'
        '가능하면 답변 끝에 참고한 페이지 번호등의 출처를 명시하세요.'
    )
)

response = agent.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': "학생 관리에 대해서 알려줘."
            }
        ]
    }
)

[TOOL START]retrieve_context/args={'query': '학생 관리'}
[TOOL END]retrieve_context/result=('[문서 1]\nsource: ./data/wt_data.pdf\npage: 1\ncontent: 방학 중 평일\n10:00 - 17:00\n매주 수요일은 16:00 종료\n토요일\n09:30 - 12:30\n학기 중에만 운영\n일요일 및 공휴일\n운영하지 않음\n온라인 FAQ만 이용 가능\n단, 시험기간 마지막 2주에는 학업상담 수요가 증가하므로 야간상담을 화요일과 목요일에 19:30까지 한시적으로\n운영한다. 야간상담은 사전 예약자만 가능하며 당일 접수는 불가하다.\n1-2. 위치와 연락처\n학생회관 2층 204호에 위치하며, 대표전화는 02-000-1234, 대표 이메일은 support@example.ac.kr 이다.\n심리상담실은 같은 층 209호에 별도로 있다.\n\n[문서 2]\nsource: ./data/wt_data.pdf\npage: 5\ncontent: 5-1. 추천 평가 질문\n•\n장학금 서류 보완은 마감 이후에도 가능한가?\n•\n상담 노쇼 예외가 인정되려면 무엇이 필요한가?\n•\n야간상담은 언제 운영되며 당일 접수가 가능한가?\n•\n비교과 프로그램의 온라인 특강 출석 기준은 몇 퍼센트인가?\n문서 끝. 이 PDF는 교육용 예시이며 실제 학교 정책이 아니다.\n\n[문서 3]\nsource: ./data/wt_data.pdf\npage: 0\ncontent: •\n장학금, 상담, 비교과 프로그램 등 여러 섹션 중 관련 영역만 정확히 검색하기\n•\n표에 있는 운영시간과 본문에 있는 예외규정을 함께 결합해 답변하기\n•\n청크 크기와 overlap에 따라 검색 품질이 어떻게 달라지는지 비교하기\n버전 1.0 / 발행일 2026-04-14 / 발행처 예시대학교 학생지원센터\n\n[문서 4]\nsource: ./data/wt_data.pdf\npage: 1\ncontent: Page 

In [74]:
print(f"{response['messages'][-1].content}")

예시대학교 학생지원센터에서 제공하는 주요 서비스와 대상은 다음과 같습니다.

**1. 핵심 서비스**
학생지원센터는 학업, 진로, 심리상담, 비교과 프로그램, 기초 행정 문의를 통합적으로 지원합니다. 구체적인 서비스 항목은 다음과 같습니다.
*   학업상담 및 학습코칭
*   심리상담 (심리상담실은 학생회관 2층 209호에 별도 위치)
*   장학금 안내
*   비교과 프로그램 신청 지원
*   기초 진로설계 및 멘토링 매칭
*   증명서 발급 안내

**2. 서비스 이용 대상**
*   **재학생:** 모든 서비스를 기본 대상으로 운영합니다.
*   **휴학생:** 일부 서비스만 이용이 가능합니다. (예: 장학금의 경우 일반적인 안내는 가능하나, 실제 신청은 장학금 종류에 따라 재학생 기준이 적용될 수 있음)
*   **졸업생:** 진로상담과 취업특강 자료 열람은 가능하지만, 장학금 상담은 지원 대상에서 제외됩니다.

참고: [문서 4, 페이지 2], [문서 5, 페이지 3]
